# Read and Write

Read and write one S3 object: bytes and text, metadata and tags on write, conditional writes, empty objects and folder markers, file objects with `open` (JSON, CSV, gzip, streaming), and uploading local files.

## Setup

This cell only prepares the notebook: it connects to the S3 bucket that the s3pathlib CI uses and empties the folder of this chapter, so every run starts from the same state. In your own code you only need the two lines marked below, with your own AWS profile and bucket.
`show_error` prints an exception in one line, so the demos can show what goes wrong.

In [1]:
import os

import boto3
from s3pathlib.api import S3Path
from s3pathlib.api import s3_context

boto_ses = boto3.Session(profile_name=os.environ["S3PATHLIB_TEST_AWS_PROFILE"])
s3_context.attach_s3_client(boto_ses.client("s3"))  # <-- in your code
aws_account_id = boto_ses.client("sts").get_caller_identity()["Account"]
bucket = f"{aws_account_id}-{boto_ses.region_name}-s3pathlib-test"  # <-- your bucket

# every demo of this chapter writes under this folder
s3dir_chapter = S3Path(bucket, "projects/s3pathlib/docs/03-Read-and-Write/")
_ = s3dir_chapter.delete()


def show_error(func):
    try:
        func()
    except Exception as e:
        print(f"{type(e).__name__}: {e}")

## 1. Bytes and text

`write_bytes` / `write_text` replace the object if it exists, and return a **new** `S3Path` with a snapshot of what was written (`size`, `etag`, ...), so reading those costs no AWS call. The path you called it on does not change; see [Metadata and Tags](../05-Metadata-and-Tags/index.ipynb) for the snapshot model.

In [2]:
s3dir = s3dir_chapter / "bytes-and-text/"
p = s3dir / "hello.txt"
p_new = p.write_bytes(b"Hello, S3!")
p_new.size, p_new.etag

(10, '9176e88e2973b7e2e260c438f56b1cd0')

In [3]:
p.read_bytes(), p.read_text()

(b'Hello, S3!', 'Hello, S3!')

`encoding` works as in `str.encode` / `bytes.decode`:

In [4]:
p = s3dir / "greeting.txt"
_ = p.write_text("你好", encoding="utf-16")
p.read_bytes(), p.read_text(encoding="utf-16")

(b'\xff\xfe`O}Y', '你好')

`range` reads part of the object, with the syntax of the HTTP `Range` header (both ends included):

In [5]:
(s3dir / "hello.txt").read_bytes(range="bytes=0-4")

b'Hello'

Reading a missing object raises boto3's own error as is (`NoSuchKey`); catch it as `botocore.exceptions.ClientError` and check `e.response["Error"]["Code"]`, or with `s3_client.exceptions.NoSuchKey`. A path that is not an object, such as a bucket, is rejected before any AWS call.

In [6]:
show_error(lambda: (s3dir / "missing.txt").read_text())
show_error(lambda: S3Path(bucket).read_text())

NoSuchKey: An error occurred (NoSuchKey) when calling the GetObject operation: The specified key does not exist.
S3PathTypeError: 's3://123456789012-us-east-1-s3pathlib-test/' is a bucket, it has no object metadata!


## 2. Metadata, tags and headers

The other arguments of `write_bytes` / `write_text` are the arguments of boto3's [put_object](https://docs.aws.amazon.com/boto3/latest/reference/services/s3/client/put_object.html) in snake case. `metadata` and `tags` take a plain `dict`.

In [7]:
s3dir = s3dir_chapter / "metadata-and-tags/"
p = s3dir / "report.json"
p_new = p.write_text(
    '{"total": 42}',
    content_type="application/json",
    metadata={"author": "alice"},
    tags={"project": "docs"},
)
p_new.metadata

{'author': 'alice'}

In [8]:
version_id, tags = p.get_tags()  # tags of the latest version
p.head_object()["ContentType"], tags

('application/json', {'project': 'docs'})

S3 stores the keys of user metadata in lower case, so `{"Author": ...}` comes back as `{"author": ...}`; s3pathlib warns when you write an upper case key.

## 3. Conditional writes

`if_none_match="*"` only writes when the object does not exist, in one atomic call: two processes that write at the same time cannot both succeed. The losing call raises `PreconditionFailed`.

In [9]:
s3dir = s3dir_chapter / "conditional-writes/"
p = s3dir / "lock.txt"
_ = p.write_text("owner: job-1", if_none_match="*")
show_error(lambda: p.write_text("owner: job-2", if_none_match="*"))
p.read_text()

ClientError: An error occurred (PreconditionFailed) when calling the PutObject operation: At least one of the pre-conditions you specified did not hold


'owner: job-1'

`if_match=<etag>` only replaces the object if nobody changed it since you last saw it (optimistic locking): keep the ETag of what you read or wrote, and send it back with the next write.

In [10]:
p = s3dir / "counter.txt"
etag = p.write_text("1").etag  # the version we know
_ = p.write_text("100")  # someone else writes in between

show_error(lambda: p.write_text("2", if_match=etag))
p.read_text()

ClientError: An error occurred (PreconditionFailed) when calling the PutObject operation: At least one of the pre-conditions you specified did not hold


'100'

## 4. Empty objects and folder markers

`touch` creates an empty object, and never changes an existing one: it raises `S3FileAlreadyExist`, or does nothing with `exist_ok=True`.

In [11]:
s3dir = s3dir_chapter / "touch-and-mkdir/"
p = s3dir / "_SUCCESS"
p.touch()
show_error(lambda: p.touch())
p.touch(exist_ok=True)
p.read_bytes()

S3FileAlreadyExist: S3 object 's3://123456789012-us-east-1-s3pathlib-test/projects/s3pathlib/docs/03-Read-and-Write/touch-and-mkdir/_SUCCESS' already exists!


b''

S3 has no folders. A folder "exists" as soon as one key starts with it. `mkdir` writes a *folder marker*, the empty object `folder/` that "Create folder" makes in the S3 console, so that an empty folder shows up; `parents=True` also writes the markers of the parent folders.

In [12]:
s3dir_new = s3dir / "a/b/c/"
print(s3dir_new.exists())
s3dir_new.mkdir(parents=True)
print(s3dir_new.exists())
[p - s3dir for p in s3dir.iter_objects()], s3dir.count_objects(include_folder=True)

False
True


([S3RelPath('_SUCCESS')], 5)

`iter_objects` skips folder markers, and `count_objects` only counts them with `include_folder=True`: the 5 objects are `_SUCCESS` and the markers `touch-and-mkdir/`, `a/`, `a/b/` and `a/b/c/`. Note that `parents=True` writes the markers of **every** parent up to the bucket (here also `projects/`, `projects/s3pathlib/`, ...), not only of the folders that did not exist: in S3 a folder without marker "exists" too.

`mkdir` on an existing folder raises `S3FolderAlreadyExist`, unless `exist_ok=True`:

In [13]:
show_error(lambda: s3dir_new.mkdir())
s3dir_new.mkdir(exist_ok=True)

S3FolderAlreadyExist: S3 folder 's3://123456789012-us-east-1-s3pathlib-test/projects/s3pathlib/docs/03-Read-and-Write/touch-and-mkdir/a/b/c/' already exists!


## 5. Exists

`exists` works on a file, a folder (any key under it, a folder marker included) and a bucket. `ensure_not_exists` raises if the path exists, handy before an operation that must not replace anything.

In [14]:
s3dir = s3dir_chapter / "exists/"
p = s3dir / "data.csv"
_ = p.write_text("a,b")
print(p.exists(), s3dir.exists(), S3Path(bucket).exists())
print((s3dir / "missing.csv").exists(), (s3dir / "missing/").exists())
show_error(p.ensure_not_exists)

True True True
False False
S3FileAlreadyExist: S3 object 's3://123456789012-us-east-1-s3pathlib-test/projects/s3pathlib/docs/03-Read-and-Write/exists/data.csv' already exists!


## 6. File objects with `open`

`open` returns a file object (through [smart_open](https://github.com/piskvorky/smart_open)), so any library that reads or writes files works with S3. The content is streamed: a big object is never loaded in memory at once. Always use it in a `with` block; a write is only finished when the file is closed.

In [15]:
import json

s3dir = s3dir_chapter / "open/"
p = s3dir / "config.json"
with p.open("w") as f:
    json.dump({"name": "alice", "tags": ["a", "b"]}, f)

with p.open("r") as f:
    data = json.load(f)
data

{'name': 'alice', 'tags': ['a', 'b']}

Read a big file line by line:

In [16]:
p = s3dir / "events.log"
with p.open("w") as f:
    for i in range(1000):
        f.write(f"event {i}\n")

n_lines = 0
with p.open("r") as f:
    for line in f:
        n_lines += 1
n_lines, line

(1000, 'event 999\n')

The `csv` module needs `newline=""`, as with local files:

In [17]:
import csv

p = s3dir / "users.csv"
with p.open("w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "name"])
    writer.writerow([1, "alice"])

with p.open("r", newline="") as f:
    rows = list(csv.DictReader(f))
rows

[{'id': '1', 'name': 'alice'}]

**Compression follows the extension.** Writing `*.gz` (or `.bz2`, `.zst`, ...) writes compressed data, and reading it gives the uncompressed content. `read_bytes` shows what is really stored: gzip data starts with the bytes `1f 8b`. Use `compression="disable"` to read or write the bytes as they are.

In [18]:
p = s3dir / "data.json.gz"
with p.open("w") as f:
    json.dump({"compressed": True}, f)

with p.open("r") as f:
    print(json.load(f))
print(p.read_bytes()[:2].hex())
with p.open("rb", compression="disable") as f:
    print(f.read(2).hex())

{'compressed': True}
1f8b
1f8b


If an exception is raised inside the `with` block of a write, the upload is aborted and nothing is written, so there is never a half written object:

In [19]:
p = s3dir / "half-written.txt"
try:
    with p.open("w") as f:
        f.write("first part")
        raise RuntimeError("the job failed")
except RuntimeError:
    pass
p.exists()

False

`open` also takes the write arguments of `write_bytes` (`metadata`, `tags`, `content_type`, ...). By default it uploads in parts while you write (multipart upload); `multipart_upload=False` keeps the data in memory and sends it with one `put_object` when the file is closed, which is required for the arguments that only `put_object` has, such as `if_none_match`.

## 7. Upload local files

`upload_file` uploads one local file (with boto3's managed transfer, in parts for a big file) and returns the new `S3Path`. Like the copy methods, it does not replace an existing object unless `overwrite=True`.

In [20]:
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()
dir_local = Path(tmp.name)
(dir_local / "report.txt").write_text("local report")
(dir_local / "logs").mkdir()
(dir_local / "logs" / "1.log").write_text("log 1")
(dir_local / "logs" / "2.log").write_text("log 2")

s3dir = s3dir_chapter / "upload/"
p_new = (s3dir / "report.txt").upload_file(dir_local / "report.txt")
print(p_new.read_text())
show_error(lambda: (s3dir / "report.txt").upload_file(dir_local / "report.txt"))

local report
S3FileAlreadyExist: S3 object 's3://123456789012-us-east-1-s3pathlib-test/projects/s3pathlib/docs/03-Read-and-Write/upload/report.txt' already exists!


`upload_dir` uploads every file of a local directory (or the files matching a glob `pattern`), keeping their relative paths, and returns how many it uploaded. With `overwrite=False` (the default) it checks every target first and uploads nothing if one exists.

In [21]:
n = (s3dir / "backup/").upload_dir(dir_local, pattern="**/*.log")
print(n)
print([p - s3dir for p in s3dir.iter_objects()])
tmp.cleanup()

2
[S3RelPath('backup/logs/1.log'), S3RelPath('backup/logs/2.log'), S3RelPath('report.txt')]


## Common pitfalls

- **`write_*` returns a new path.** `p.write_text(...)` does not update the snapshot of `p`; use the returned path to read `size` / `etag` of what you wrote.
- **`open` without `with`.** The object is only written when the file is closed.
- **Compression by extension.** `open` on `data.gz` decompresses; pass `compression="disable"` to get the raw bytes.
- **Conditional writes in `open`** need `multipart_upload=False`, otherwise `ValueError`.

## See also

- [Metadata and Tags](../05-Metadata-and-Tags/index.ipynb): the snapshot model, `head_object`, tags.
- [Copy, Move and Delete](../06-Copy-Move-Delete/index.ipynb).
- [Versioning](../07-Versioning/index.ipynb): read and write a specific version (`version_id`).